In [1]:
path = "/home/yamane/helixEncoder/data/new_classA_bp/"

In [2]:
import pandas as pd

file_path = "/data/yamane/GPCRclassAData/"
fle_name = ["chemblSmiles.csv","classA_ligand_binary_202110.csv","id2seq.csv"]


smiles_df = pd.read_csv(file_path+fle_name[0]).drop("Unnamed: 0",axis = 1)
origin_df = pd.read_csv("origin.csv").drop("Unnamed: 0",axis = 1)
seq_df = pd.read_csv(file_path+fle_name[2]).drop("Unnamed: 0",axis = 1)

print("interaction : ",len(origin_df))
print("protein_num : ",len(set(origin_df["UniProt ID"].to_list())))
print("ligand_num : ",len(set(origin_df["Database Ligand ID"].to_list())))

interaction :  337115
protein_num :  504
ligand_num :  143807


In [76]:
# make 3 test data

cluster1 = []
cluster2 = []
cluster3 = []
cluster4 = []
for i,v in zip(list(origin_df["UniProt ID"].value_counts().index),list(origin_df["UniProt ID"].value_counts().values)):
  if v < 3000 and v > 2000:
    cluster1.append(i)
  elif v < 2000 and v > 1000:
    cluster2.append(i)
  elif v < 1000 and v > 500:
    cluster3.append(i)
  elif v < 500  and v > 100:
    cluster4.append(i)

def get_ID(cluster):
  divide = len(cluster)//3
  test1 = cluster[:divide]
  test2 = cluster[divide:2*divide]
  test3 = cluster[2*divide:]
  return test1,test2,test3

a_test1,a_test2,a_test3 = get_ID(cluster1)
b_test1,b_test2,b_test3 = get_ID(cluster2)
c_test1,c_test2,c_test3 = get_ID(cluster3)
d_test1,d_test2,d_test3 = get_ID(cluster4)

a_test1.extend(b_test1)
a_test1.extend(c_test2)
a_test1.extend(d_test3)

a_test2.extend(b_test2)
a_test2.extend(c_test3)
a_test2.extend(d_test1)

a_test3.extend(b_test3)
a_test3.extend(c_test1)
a_test3.extend(d_test2)


In [77]:
# s = 0
# test_index = []
# for i,v in zip(list(origin_df["UniProt ID"].value_counts().index),list(origin_df["UniProt ID"].value_counts().values)):
#   if v > 1000 and v < 1600:
#     s += v
#     test_index.append(i)
#     #print(i,v)
# print(s)

def make_csv_file(test_index,test_type,file_name = None):
#testdata daraframe
  for i,index in enumerate(test_index):
    if i == 0:
      test_df = origin_df[origin_df["UniProt ID"] == index]
    else:
      test_df = pd.concat([test_df,origin_df[origin_df["UniProt ID"] == index]])

  test_df = test_df.reset_index().drop("index",axis = 1)
  
  for i,name in enumerate(test_index):
    if i == 0:
      train_df = origin_df[origin_df["UniProt ID"] != name]
    else:
      train_df = train_df[train_df["UniProt ID"] != name]

  train_df = train_df.reset_index().drop("index",axis = 1)

  
  train_df.to_csv(path+test_type+"/"+"train"+file_name+".csv")
  test_df.to_csv(path+test_type+"/"+"test"+file_name+".csv")

  return train_df,test_df


In [86]:
def print_df_data(df):
  protein_num = len(set(df["UniProt ID"].to_list()))
  interaction = len(df)
  pos = df["Interaction_type"].value_counts()[1]
  neg = df["Interaction_type"].value_counts()[0]

  print("p num : ",protein_num)
  print("pos",pos/interaction)
  print("neg",neg/interaction)
  
  # print("interaction : ",len(df))
  # print("protein_num : ",len(set(df["UniProt ID"].to_list())))
  # print("ligand_num : ",len(set(df["Database Ligand ID"].to_list())))
  # print("positive",df["Interaction_type"].value_counts()[1])
  # print("negative",df["Interaction_type"].value_counts()[0])
  print("-"*20)

In [87]:
print("test1")
train_df_1,test_df_1 =  make_csv_file(a_test1,"test1","1")
# print_df_data(train_df_1)
# print_df_data(test_df_1)

print("\n\ntest2")
train_df_2,test_df_2 =  make_csv_file(a_test2,"test2","2")
# print_df_data(train_df_2)
# print_df_data(test_df_2)

print("\n\ntest3")
train_df_3,test_df_3 =  make_csv_file(a_test3,"test3","3")
# print_df_data(train_df_3)
# print_df_data(test_df_3)


test1
p num :  424
pos 0.2903938140360945
neg 0.7096061859639056
--------------------
p num :  80
pos 0.38747531643520766
neg 0.6125246835647923
--------------------


test2
p num :  424
pos 0.290876673569991
neg 0.7091233264300091
--------------------
p num :  80
pos 0.3837170450748596
neg 0.6162829549251404
--------------------


test3
p num :  423
pos 0.3053859651666981
neg 0.6946140348333019
--------------------
p num :  81
pos 0.32078581102028164
neg 0.6792141889797184
--------------------


In [52]:
def change_format(df):
  tcpi_format = []
  for i in range(len(df)):
    cid = df.iloc[i]["Database Ligand ID"]
    smiles = smiles_df[smiles_df["CHEMBL_ID"] == cid]["smiles"].values[0]
    pid = df.iloc[i]["UniProt ID"]
    seq = seq_df[seq_df["UniProt ID"] == pid]["sequence"].values[0]
    it = df.iloc[i]["Interaction_type"]
    tcpi_format.append(" ".join([smiles,seq,str(it)]))
  return tcpi_format

In [53]:
tcpi_format_train1 = change_format(train_df_1)
print("train dataset done")
tcpi_format_test1 = change_format(test_df_1)
print("test dataset done")
with open(path+"test1/"+"byProtein_train_1.txt","w") as f:
  f.write('\n'.join(tcpi_format_train1))

with open(path+"test1/"+"byProtein_test_1.txt","w") as f:
  f.write('\n'.join(tcpi_format_test1))



In [ ]:
tcpi_format_train2 = change_format(train_df_2)
print("train dataset done")
tcpi_format_test2 = change_format(test_df_2)
print("test dataset done")
with open(path+"test2/"+"byProtein_train_2.txt","w") as f:
  f.write('\n'.join(tcpi_format_train2))

with open(path+"test2/"+"byProtein_test_2.txt","w") as f:
  f.write('\n'.join(tcpi_format_test2))



In [ ]:
tcpi_format_train3 = change_format(train_df_3)
print("train dataset done")
tcpi_format_test3 = change_format(test_df_3)
print("test dataset done")
with open(path+"test3/"+"byProtein_train_3.txt","w") as f:
  f.write('\n'.join(tcpi_format_train3))

with open(path+"test3/"+"byProtein_test_3.txt","w") as f:
  f.write('\n'.join(tcpi_format_test3))